# ============================================================
# EVALUATION SUITE — FULL CV VERSION
# ============================================================
# Extends the single-fold suite to:
#   • TASKS  = ["5class", "binary"]  — both in one run
#   • RUN_FOLDS = range(5)           — full 5-fold CV per task
# Per-fold outputs go to  eval_{task}/fold{i}/
# CV-aggregated outputs go to  eval_{task}/cv_summary/
# Final comparison plot saved to  eval_combined/
#
# Runtime estimate (Colab T4):
#   5class: ~2.2 h   |   binary: ~2.1 h   |   total: ~4.3 h
#   Quick sanity: set RUN_FOLDS=[0] and TASKS=["5class"]
# ============================================================

In [ ]:
import os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from scipy import stats as scipy_stats

warnings.filterwarnings("ignore")
np.random.seed(42)

# ============================================================
# 0. CONFIGURATION
# ============================================================
SEED         = 42
DATA_DIR     = "/content/drive/MyDrive/TUMC_dataset"
TASKS        = ["5class", "binary"]
RUN_FOLDS    = list(range(5))     # [0] for quick sanity check
DROP_LEAKAGE = True
FN_COST, FP_COST = 10, 1         # security: missing attack costs 10×

LEAKY_FEATURES = {"is_tr_domain","is_https","cluster_malicious_ratio","campaign_token_score"}
META_COLS = {"url","source","tld","label","label_enc","class_final","class_enc","fold","reg_domain"}

for fname in ["features_full_final_inductive_dedup.csv",
              "features_full_final_inductive.csv","features_full_final.csv"]:
    INPUT_FILE = os.path.join(DATA_DIR, fname)
    if os.path.exists(INPUT_FILE): break

COMBINED_DIR = os.path.join(DATA_DIR, "journal_eval_combined")
os.makedirs(COMBINED_DIR, exist_ok=True)

# colour palette — consistent across all figures
MODEL_COLORS = {
    "HistGB":"#1565C0","LightGBM":"#1E88E5","XGBoost":"#42A5F5",
    "RandomForest":"#2E7D32","ExtraTrees":"#66BB6A",
    "LogisticRegression":"#F57F17","ComplementNB":"#B71C1C",
}
MODEL_ORDER = ["HistGB","LightGBM","XGBoost","RandomForest",
               "ExtraTrees","LogisticRegression","ComplementNB"]
CLASS_COLORS = ["#1565C0","#B71C1C","#2E7D32","#F57F17","#6A1B9A"]

plt.rcParams.update({
    "font.family":"serif","font.size":10,"axes.labelsize":10,
    "axes.titlesize":11,"legend.fontsize":8.5,"figure.dpi":150,
    "savefig.dpi":600,"pdf.fonttype":42,"ps.fonttype":42,
    "lines.linewidth":1.6,"axes.linewidth":0.8,"axes.spines.top":False,
    "axes.spines.right":False,"axes.grid":True,"grid.alpha":0.3,
    "grid.linestyle":"--",
})

# ============================================================
# 1. PACKAGES
# ============================================================
HAS_XGB = HAS_LGB = False
try:    import xgboost  as xgb; HAS_XGB = True; print("XGBoost  ✓")
except: print("XGBoost unavailable")
try:    import lightgbm as lgb; HAS_LGB = True; print("LightGBM ✓")
except: print("LightGBM unavailable")

from sklearn.pipeline import Pipeline
from sklearn.impute   import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, label_binarize
from sklearn.ensemble  import (HistGradientBoostingClassifier,
                               RandomForestClassifier, ExtraTreesClassifier)
from sklearn.linear_model  import LogisticRegression
from sklearn.naive_bayes   import ComplementNB
from sklearn.calibration   import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, log_loss, brier_score_loss,
    det_curve)

# ============================================================
# 2. HELPERS
# ============================================================
def safe_name(s):
    for c in " /\\:()": s = s.replace(c,"_")
    return s

def makedirs(*paths):
    for p in paths: os.makedirs(p, exist_ok=True)

def save_fig(fig, *paths):
    for p in paths:
        ext = os.path.splitext(p)[1].lower()
        fig.savefig(p, dpi=600 if ext==".png" else None, bbox_inches="tight")
    plt.close(fig)

def save_tex(df, path, caption, label, index=True, fmt="%.4f"):
    with open(path,"w",encoding="utf-8") as f:
        f.write(df.to_latex(index=index, escape=False,
                            caption=caption, label=label, float_format=fmt))

def get_class_names(df, task, classes):
    if task=="binary":
        try:
            m = df[["label_enc","label"]].drop_duplicates().sort_values("label_enc")
            if len(m)==len(classes): return m["label"].astype(str).tolist()
        except: pass
        return ["benign","malicious"]
    try:
        m = df[["class_enc","class_final"]].drop_duplicates().sort_values("class_enc")
        if len(m)==len(classes): return m["class_final"].astype(str).tolist()
    except: pass
    return (["benign","phishing","malware","scam","other_malicious"]
            if len(classes)==5 else [f"class_{c}" for c in classes])

def align_proba(model, proba, expected):
    if proba is None: return None
    try:
        mc = getattr(model,"classes_",None)
        if mc is None: mc = list(model.named_steps.values())[-1].classes_
        mc = np.array(mc); expected = np.array(expected)
        if list(mc)==list(expected): return proba
        out = np.zeros((proba.shape[0], len(expected)))
        for i,cls in enumerate(expected):
            idx = np.where(mc==cls)[0]
            if len(idx): out[:,i] = proba[:,idx[0]]
        return out
    except: return proba

# ============================================================
# 3. MODEL FACTORY
# ============================================================
def make_models(n_classes):
    is_bin = n_classes==2
    M = {}
    M["HistGB"] = HistGradientBoostingClassifier(
        max_depth=6, learning_rate=0.03, max_iter=500, random_state=SEED)
    M["RandomForest"] = Pipeline([("imp",SimpleImputer(strategy="median")),
        ("m",RandomForestClassifier(n_estimators=300, max_depth=12,
         min_samples_split=10, min_samples_leaf=5,
         class_weight="balanced", random_state=SEED, n_jobs=-1))])
    M["ExtraTrees"] = Pipeline([("imp",SimpleImputer(strategy="median")),
        ("m",ExtraTreesClassifier(n_estimators=300, max_depth=12,
         min_samples_split=10, min_samples_leaf=5, max_features="sqrt",
         bootstrap=True, class_weight="balanced", random_state=SEED, n_jobs=-1))])
    M["LogisticRegression"] = Pipeline([("imp",SimpleImputer(strategy="median")),
        ("sc",StandardScaler()),
        ("m",LogisticRegression(C=1.0, max_iter=1000,
         class_weight="balanced", random_state=SEED, n_jobs=-1))])
    M["ComplementNB"] = Pipeline([("imp",SimpleImputer(strategy="median")),
        ("mm",MinMaxScaler()),("m",ComplementNB(alpha=1.0))])
    if HAS_XGB:
        kw = dict(max_depth=4,learning_rate=0.03,n_estimators=500,
                  subsample=0.8,colsample_bytree=0.8,reg_alpha=1,reg_lambda=2,
                  random_state=SEED,n_jobs=-1,tree_method="hist")
        M["XGBoost"] = xgb.XGBClassifier(
            objective="binary:logistic" if is_bin else "multi:softprob",
            eval_metric="logloss" if is_bin else "mlogloss",
            **({"scale_pos_weight":5.5} if is_bin else {"num_class":n_classes}), **kw)
    if HAS_LGB:
        kw = dict(num_leaves=31,max_depth=6,learning_rate=0.03,n_estimators=500,
                  feature_fraction=0.8,bagging_fraction=0.8,bagging_freq=5,
                  reg_alpha=1,reg_lambda=1,class_weight="balanced",
                  random_state=SEED,n_jobs=-1,verbose=-1)
        M["LightGBM"] = lgb.LGBMClassifier(
            objective="binary" if is_bin else "multiclass",
            **({"num_class":n_classes} if not is_bin else {}), **kw)
    return M

# ============================================================
# 4. SINGLE-FOLD EVALUATION  — returns metrics + saves preds
# ============================================================
def eval_one_fold(name, model, X_tr, y_tr, X_te, y_te,
                  task, classes, class_names, test_df,
                  csv_dir, tex_dir, png_dir, pdf_dir, fold_id):
    is_bin = len(classes)==2
    mid    = safe_name(name)

    t0 = time.perf_counter(); model.fit(X_tr,y_tr); train_t = time.perf_counter()-t0
    t1 = time.perf_counter(); y_pred = model.predict(X_te); pred_t = time.perf_counter()-t1

    y_proba = None
    if hasattr(model,"predict_proba"):
        try: y_proba = align_proba(model, model.predict_proba(X_te), classes)
        except: pass

    # save for ROC/PR plotting
    np.save(os.path.join(csv_dir,f"y_true_{mid}.npy"), y_te)
    np.save(os.path.join(csv_dir,f"y_pred_{mid}.npy"), y_pred)
    if y_proba is not None:
        np.save(os.path.join(csv_dir,f"y_proba_{mid}.npy"), y_proba)

    # save error URLs for error-analysis
    err_mask = y_pred != y_te
    if "url" in test_df.columns:
        err_df = test_df[err_mask][["url","class_final","source","tld"]].copy() \
                 if task=="5class" else test_df[err_mask][["url","label","source","tld"]].copy()
        err_df["predicted"] = y_pred[err_mask]
        err_df.to_csv(os.path.join(csv_dir,f"errors_{mid}.csv"), index=False)

    # metrics
    f1m  = f1_score(y_te,y_pred,average="macro",     zero_division=0)
    f1w  = f1_score(y_te,y_pred,average="weighted",  zero_division=0)
    prm  = precision_score(y_te,y_pred,average="macro",  zero_division=0)
    rem  = recall_score(y_te,y_pred,   average="macro",  zero_division=0)
    mcc  = matthews_corrcoef(y_te,y_pred)
    kap  = cohen_kappa_score(y_te,y_pred)
    acc  = accuracy_score(y_te,y_pred)
    bacc = balanced_accuracy_score(y_te,y_pred)

    roc_ovr=roc_ovo=pr_auc=ll=brier=np.nan
    if y_proba is not None:
        try: ll = log_loss(y_te,y_proba,labels=classes)
        except: pass
        if is_bin:
            p1 = y_proba[:,1]
            try: roc_ovr = roc_auc_score(y_te,p1)
            except: pass
            try: pr_auc  = average_precision_score(y_te,p1)
            except: pass
            try: brier   = brier_score_loss(y_te,p1)
            except: pass
        else:
            try: roc_ovr = roc_auc_score(y_te,y_proba,multi_class="ovr",average="macro")
            except: pass
            try: roc_ovo = roc_auc_score(y_te,y_proba,multi_class="ovo",average="macro")
            except: pass
            try:
                yb = label_binarize(y_te,classes=classes)
                pr_auc = average_precision_score(yb,y_proba,average="macro")
            except: pass

    # binary extras
    cm = confusion_matrix(y_te,y_pred,labels=classes)
    bin_m = {}
    if is_bin and cm.shape==(2,2):
        tn,fp,fn,tp = cm.ravel()
        bin_m = {"FPR":fp/(fp+tn) if fp+tn else 0,"FNR":fn/(fn+tp) if fn+tp else 0,
                 "FN":int(fn),"FP":int(fp),"TP":int(tp),"TN":int(tn),
                 "WeightedCost":int(fn*FN_COST+fp*FP_COST)}

    print(f"  F1={f1m:.4f}  MCC={mcc:.4f}  "
          f"AUC={roc_ovr:.4f}  PR={pr_auc:.4f}  ({train_t:.0f}s)")
    return {"Model":name,"Fold":fold_id,"Accuracy":acc,"Balanced_Accuracy":bacc,
            "Precision_Macro":prm,"Recall_Macro":rem,"F1_Macro":f1m,"F1_Weighted":f1w,
            "MCC":mcc,"Cohen_Kappa":kap,"ROC_AUC_OVR":roc_ovr,"ROC_AUC_OVO":roc_ovo,
            "PR_AUC":pr_auc,"Log_Loss":ll,"Brier_Score":brier,
            "Train_Time_s":round(train_t,1),"Pred_Time_s":round(pred_t,2),
            "Errors":int(err_mask.sum()), **bin_m}

# ============================================================
# 5. CV AGGREGATION
# ============================================================
def aggregate_cv(rows):
    df = pd.DataFrame(rows)
    num = [c for c in df.select_dtypes(include=np.number).columns
           if c not in ("Fold","Errors","FN","FP","TP","TN","WeightedCost")]
    res = {"Model":rows[0]["Model"]}
    for c in num:
        res[f"{c}_mean"] = df[c].mean()
        res[f"{c}_std"]  = df[c].std()
    res["Total_Errors"] = df["Errors"].sum()
    return res

# ============================================================
# 6A. ROC CURVES  (all models, one figure per task)
# ============================================================
def plot_roc_curves(fold0_dir, task, classes, class_names,
                    is_bin, png_dir, pdf_dir):
    fig_h = 5 if is_bin else max(6, 2.5*len(classes))
    n_sub = 1 if is_bin else len(classes)
    fig, axes = plt.subplots(1, n_sub, figsize=(6*n_sub if not is_bin else 7, fig_h))
    if is_bin: axes = [axes]

    for mn in MODEL_ORDER:
        mid = safe_name(mn)
        fpr_p = os.path.join(fold0_dir,f"y_proba_{mid}.npy")
        ftr_p = os.path.join(fold0_dir,f"y_true_{mid}.npy")
        if not (os.path.exists(fpr_p) and os.path.exists(ftr_p)): continue
        y_proba = np.load(fpr_p); y_true = np.load(ftr_p)
        col = MODEL_COLORS.get(mn,"#607D8B")
        if is_bin:
            fpr,tpr,_ = roc_curve(y_true, y_proba[:,1])
            auc = roc_auc_score(y_true, y_proba[:,1])
            axes[0].plot(fpr, tpr, color=col, label=f"{mn} (AUC={auc:.3f})")
        else:
            yb = label_binarize(y_true, classes=classes)
            for ci, (ax, cname) in enumerate(zip(axes, class_names)):
                try:
                    fpr,tpr,_ = roc_curve(yb[:,ci], y_proba[:,ci])
                    auc = roc_auc_score(yb[:,ci], y_proba[:,ci])
                    ax.plot(fpr, tpr, color=col, label=f"{mn} ({auc:.3f})")
                    ax.set_title(cname, fontweight="bold")
                except: pass

    for ax in axes:
        ax.plot([0,1],[0,1],"k--",lw=0.8,label="Random")
        ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
        ax.legend(fontsize=7 if not is_bin else 8.5)
    axes[0].set_title("ROC Curves — " + task, fontweight="bold")
    fig.suptitle("Receiver Operating Characteristic", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"roc_curves.png"),
                  os.path.join(pdf_dir,"roc_curves.pdf"))
    print("  ✓ ROC curves")

# ============================================================
# 6B. PRECISION-RECALL CURVES
# ============================================================
def plot_pr_curves(fold0_dir, task, classes, class_names,
                   is_bin, png_dir, pdf_dir):
    n_sub = 1 if is_bin else len(classes)
    fig, axes = plt.subplots(1, n_sub, figsize=(6*n_sub if not is_bin else 7, 5))
    if is_bin: axes = [axes]

    for mn in MODEL_ORDER:
        mid = safe_name(mn)
        fp = os.path.join(fold0_dir,f"y_proba_{mid}.npy")
        ft = os.path.join(fold0_dir,f"y_true_{mid}.npy")
        if not (os.path.exists(fp) and os.path.exists(ft)): continue
        y_proba = np.load(fp); y_true = np.load(ft)
        col = MODEL_COLORS.get(mn,"#607D8B")
        if is_bin:
            prec,rec,_ = precision_recall_curve(y_true, y_proba[:,1])
            ap = average_precision_score(y_true, y_proba[:,1])
            axes[0].plot(rec, prec, color=col, label=f"{mn} (AP={ap:.3f})")
        else:
            yb = label_binarize(y_true, classes=classes)
            for ci,(ax,cname) in enumerate(zip(axes,class_names)):
                try:
                    prec,rec,_ = precision_recall_curve(yb[:,ci], y_proba[:,ci])
                    ap = average_precision_score(yb[:,ci], y_proba[:,ci])
                    ax.plot(rec,prec,color=col,label=f"{mn} ({ap:.3f})")
                    ax.set_title(cname, fontweight="bold")
                    # baseline = class prevalence
                    ax.axhline(yb[:,ci].mean(), ls=":", color=col, lw=0.8)
                except: pass

    for ax in axes:
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
        ax.legend(fontsize=7 if not is_bin else 8.5)
    axes[0].set_title("Precision-Recall Curves — "+task, fontweight="bold")
    fig.suptitle("Precision-Recall (critical for imbalanced classes)", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"pr_curves.png"),
                  os.path.join(pdf_dir,"pr_curves.pdf"))
    print("  ✓ PR curves")

# ============================================================
# 6C. DET CURVES  (binary only)
# ============================================================
def plot_det_curves(fold0_dir, task, is_bin, png_dir, pdf_dir):
    if not is_bin: return
    fig, ax = plt.subplots(figsize=(6.5,5.5))
    for mn in MODEL_ORDER:
        mid = safe_name(mn)
        fp = os.path.join(fold0_dir,f"y_proba_{mid}.npy")
        ft = os.path.join(fold0_dir,f"y_true_{mid}.npy")
        if not (os.path.exists(fp) and os.path.exists(ft)): continue
        y_proba = np.load(fp); y_true = np.load(ft)
        try:
            fpr,fnr,_ = det_curve(y_true, y_proba[:,1])
            ax.plot(fpr,fnr,color=MODEL_COLORS.get(mn,"#607D8B"),label=mn)
        except: pass
    ax.set_xlabel("False Positive Rate (FPR)")
    ax.set_ylabel("False Negative Rate (FNR)")
    ax.set_title("Detection Error Tradeoff (DET) — Binary", fontweight="bold")
    ax.legend()
    # EER line
    ax.plot([0,1],[0,1],"k--",lw=0.7,alpha=0.4,label="EER")
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"det_curves.png"),
                  os.path.join(pdf_dir,"det_curves.pdf"))
    print("  ✓ DET curves")

# ============================================================
# 6D. CALIBRATION CURVES
# ============================================================
def plot_calibration(fold0_dir, task, is_bin, png_dir, pdf_dir):
    n_mods = len(MODEL_ORDER)
    ncols  = 4; nrows = (n_mods+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(4.5*ncols, 4*nrows), squeeze=False)
    axes_flat = axes.flatten()

    brier_rows = []
    for i, mn in enumerate(MODEL_ORDER):
        ax = axes_flat[i]
        mid = safe_name(mn)
        fp = os.path.join(fold0_dir,f"y_proba_{mid}.npy")
        ft = os.path.join(fold0_dir,f"y_true_{mid}.npy")
        if not (os.path.exists(fp) and os.path.exists(ft)):
            ax.set_visible(False); continue
        y_proba = np.load(fp); y_true = np.load(ft)
        col = MODEL_COLORS.get(mn,"#607D8B")
        try:
            if is_bin:
                prob = y_proba[:,1]
            else:
                prob = y_proba.max(axis=1)  # confidence of predicted class
                y_true = (y_true == y_proba.argmax(axis=1)).astype(int)
            frac,mean_p = calibration_curve(y_true, prob, n_bins=10)
            ax.plot(mean_p, frac, "o-", color=col, label=mn, ms=4)
            ax.plot([0,1],[0,1],"k--",lw=0.8,label="Perfect")
            bs = brier_score_loss(y_true, prob)
            ax.set_title(f"{mn}\nBrier={bs:.4f}", fontsize=9)
            brier_rows.append({"Model":mn,"Brier":bs})
        except Exception as e:
            ax.set_title(f"{mn}\n(error)", fontsize=9)
        ax.set_xlabel("Mean predicted prob.")
        ax.set_ylabel("Fraction positives")
        ax.legend(fontsize=7)

    for j in range(i+1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(f"Calibration Reliability Diagrams — {task}",
                 fontweight="bold", fontsize=12)
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"calibration_curves.png"),
                  os.path.join(pdf_dir,"calibration_curves.pdf"))
    print("  ✓ Calibration curves")
    return pd.DataFrame(brier_rows)

# ============================================================
# 6E. CONFUSION MATRICES — all models in one figure
# ============================================================
def plot_all_confusion_matrices(fold0_dir, task, classes, class_names,
                                png_dir, pdf_dir):
    n_mods = len(MODEL_ORDER)
    ncols  = min(4, n_mods); nrows = (n_mods+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(4.8*ncols, 4.4*nrows), squeeze=False)
    axes_flat = axes.flatten()

    for i, mn in enumerate(MODEL_ORDER):
        ax = axes_flat[i]
        mid = safe_name(mn)
        fp = os.path.join(fold0_dir,f"y_pred_{mid}.npy")
        ft = os.path.join(fold0_dir,f"y_true_{mid}.npy")
        if not (os.path.exists(fp) and os.path.exists(ft)):
            ax.set_visible(False); continue
        y_pred = np.load(fp); y_true = np.load(ft)
        cm = confusion_matrix(y_true, y_pred, labels=classes, normalize="true")
        im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
        ax.set_title(mn, fontweight="bold", fontsize=9)
        ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
        short = [c[:6] for c in class_names]
        ax.set_xticklabels(short, rotation=45, ha="right", fontsize=7)
        ax.set_yticklabels(short, fontsize=7)
        thr = 0.5
        for r in range(len(classes)):
            for c2 in range(len(classes)):
                ax.text(c2, r, f"{cm[r,c2]:.2f}", ha="center", va="center",
                        color="white" if cm[r,c2]>thr else "black", fontsize=7)
        fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)

    for j in range(i+1, len(axes_flat)): axes_flat[j].set_visible(False)
    fig.suptitle(f"Row-Normalised Confusion Matrices — {task} (fold 0)",
                 fontweight="bold", fontsize=12)
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"all_confusion_matrices.png"),
                  os.path.join(pdf_dir,"all_confusion_matrices.pdf"))
    print("  ✓ Combined confusion matrices")

# ============================================================
# 6F. THRESHOLD ANALYSIS + COST (binary best model only)
# ============================================================
def plot_threshold_and_cost(fold0_dir, best_model, is_bin, png_dir, pdf_dir):
    if not is_bin: return
    mid = safe_name(best_model)
    fp = os.path.join(fold0_dir,f"y_proba_{mid}.npy")
    ft = os.path.join(fold0_dir,f"y_true_{mid}.npy")
    if not (os.path.exists(fp) and os.path.exists(ft)): return

    y_proba = np.load(fp); y_true = np.load(ft)
    prob1   = y_proba[:,1]
    thresholds = np.arange(0.01, 0.99, 0.01)

    precs,recs,f1s,specs,costs,fprs,fnrs = [],[],[],[],[],[],[]
    for th in thresholds:
        yp = (prob1 >= th).astype(int)
        cm = confusion_matrix(y_true, yp)
        tn,fp2,fn,tp = (cm.ravel() if cm.shape==(2,2)
                        else [0,0,0,0])
        precs.append(precision_score(y_true,yp,zero_division=0))
        recs.append(recall_score(y_true,yp,zero_division=0))
        f1s.append(f1_score(y_true,yp,zero_division=0))
        specs.append(tn/(tn+fp2) if tn+fp2 else 0)
        costs.append(fn*FN_COST + fp2*FP_COST)
        fprs.append(fp2/(fp2+tn) if fp2+tn else 0)
        fnrs.append(fn/(fn+tp)   if fn+tp   else 0)

    best_f1_th  = thresholds[np.argmax(f1s)]
    best_cost_th= thresholds[np.argmin(costs)]
    best_fnr_th = thresholds[np.argmin(
        [a*FN_COST + b*FP_COST for a,b in zip(fnrs,fprs)])]

    fig, axes = plt.subplots(1,2,figsize=(13,5))

    # Panel 1: metrics vs threshold
    ax = axes[0]
    ax.plot(thresholds, precs,  label="Precision",    color="#1565C0")
    ax.plot(thresholds, recs,   label="Recall",       color="#B71C1C")
    ax.plot(thresholds, f1s,    label="F1",           color="#2E7D32", lw=2)
    ax.plot(thresholds, specs,  label="Specificity",  color="#F57F17", ls="--")
    ax.axvline(best_f1_th,   color="#2E7D32", ls=":",  alpha=0.8,
               label=f"Best F1 (th={best_f1_th:.2f})")
    ax.axvline(best_cost_th, color="#6A1B9A", ls="-.", alpha=0.8,
               label=f"Min cost (th={best_cost_th:.2f})")
    ax.axvline(0.5, color="gray", ls="--", lw=0.7, label="Default (0.5)")
    ax.set_xlabel("Classification threshold"); ax.set_ylabel("Score")
    ax.set_title(f"Threshold Sensitivity — {best_model}", fontweight="bold")
    ax.legend(fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1.02)

    # Panel 2: cost vs threshold
    ax2 = axes[1]
    ax2.plot(thresholds, costs, color="#B71C1C", lw=2)
    ax2.axvline(best_cost_th, color="#6A1B9A", ls="--",
                label=f"Optimal th={best_cost_th:.2f}\n(FN×{FN_COST}+FP×{FP_COST})")
    ax2.set_xlabel("Classification threshold")
    ax2.set_ylabel(f"Total cost (FN×{FN_COST} + FP×{FP_COST})")
    ax2.set_title("Cost-Sensitive Analysis (Security Context)", fontweight="bold")
    ax2.legend(fontsize=8)
    # annotate min cost
    min_c = min(costs); min_th = thresholds[np.argmin(costs)]
    ax2.annotate(f"Min cost={min_c:,.0f}\n@th={min_th:.2f}",
                 xy=(min_th, min_c),
                 xytext=(min_th+0.1, min_c*(1.05)),
                 arrowprops=dict(arrowstyle="->",color="black"),fontsize=8)

    fig.suptitle("Threshold & Cost Analysis (Binary Detection)", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"threshold_cost_analysis.png"),
                  os.path.join(pdf_dir,"threshold_cost_analysis.pdf"))
    print(f"  ✓ Threshold analysis (best F1 th={best_f1_th:.2f}, min-cost th={best_cost_th:.2f})")

# ============================================================
# 6G. PER-CLASS HEATMAP  (5-class)
# ============================================================
def plot_per_class_heatmap(fold0_dir, task, classes, class_names,
                           is_bin, png_dir, pdf_dir):
    if is_bin: return
    metrics = ["precision","recall","f1-score"]
    data = {}
    for mn in MODEL_ORDER:
        mid = safe_name(mn)
        fp = os.path.join(fold0_dir,f"y_pred_{mid}.npy")
        ft = os.path.join(fold0_dir,f"y_true_{mid}.npy")
        if not (os.path.exists(fp) and os.path.exists(ft)): continue
        y_pred = np.load(fp); y_true = np.load(ft)
        rep = classification_report(y_true, y_pred, labels=classes,
                                    target_names=class_names,
                                    output_dict=True, zero_division=0)
        data[mn] = {cn: {m: rep[cn][m] for m in metrics}
                    for cn in class_names}

    if not data: return
    fig, axes = plt.subplots(1, len(metrics), figsize=(5.5*len(metrics), 5.5))
    mnames = [m for m in MODEL_ORDER if m in data]

    for ax, met in zip(axes, metrics):
        mat = np.array([[data[mn][cn][met] for cn in class_names]
                        for mn in mnames])
        im  = ax.imshow(mat, cmap="RdYlGn", vmin=0.5, vmax=1.0, aspect="auto")
        ax.set_xticks(range(len(class_names)))
        ax.set_xticklabels(class_names, rotation=35, ha="right", fontsize=8)
        ax.set_yticks(range(len(mnames)))
        ax.set_yticklabels(mnames, fontsize=8)
        ax.set_title(met.capitalize(), fontweight="bold")
        for r in range(len(mnames)):
            for c2 in range(len(class_names)):
                ax.text(c2,r,f"{mat[r,c2]:.2f}",ha="center",va="center",
                        fontsize=7.5,
                        color="white" if mat[r,c2]<0.6 else "black")
        fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)

    fig.suptitle("Per-Class Performance Heatmap — "+task,
                 fontweight="bold", fontsize=12)
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"per_class_heatmap.png"),
                  os.path.join(pdf_dir,"per_class_heatmap.pdf"))
    print("  ✓ Per-class heatmap")

# ============================================================
# 6H. RADAR CHART  (CV means)
# ============================================================
def plot_radar(cv_summary, task, png_dir, pdf_dir):
    RADAR_METRICS = ["F1_Macro_mean","MCC_mean","Precision_Macro_mean",
                     "Recall_Macro_mean","Balanced_Accuracy_mean","ROC_AUC_OVR_mean"]
    labels = ["F1-Macro","MCC","Precision","Recall","Bal-Acc","AUC"]
    mnames = [m for m in MODEL_ORDER if m in cv_summary]
    if len(mnames) < 2: return

    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7,7), subplot_kw={"polar":True})
    for mn in mnames:
        cv = cv_summary[mn]
        vals = [cv.get(m,0) for m in RADAR_METRICS]
        vals += vals[:1]
        ax.plot(angles, vals, "o-", color=MODEL_COLORS.get(mn,"#607D8B"),
                lw=1.6, label=mn, ms=4)
        ax.fill(angles, vals, color=MODEL_COLORS.get(mn,"#607D8B"), alpha=0.05)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
    ax.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"], fontsize=7)
    ax.set_title(f"Model Radar Chart — {task} (5-fold CV means)",
                 fontweight="bold", pad=15)
    ax.legend(loc="lower right", bbox_to_anchor=(1.35,0), fontsize=8)
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"radar_chart.png"),
                  os.path.join(pdf_dir,"radar_chart.pdf"))
    print("  ✓ Radar chart")

# ============================================================
# 6I. EFFICIENCY PLOT (F1 vs Training Time)
# ============================================================
def plot_efficiency(cv_summary, task, png_dir, pdf_dir):
    fig, ax = plt.subplots(figsize=(7.5,5))
    mnames = [m for m in MODEL_ORDER if m in cv_summary]
    for mn in mnames:
        cv  = cv_summary[mn]
        f1m = cv.get("F1_Macro_mean", np.nan)
        f1s = cv.get("F1_Macro_std",  0)
        tt  = cv.get("Train_Time_s_mean", np.nan)
        if np.isnan(f1m) or np.isnan(tt): continue
        col = MODEL_COLORS.get(mn,"#607D8B")
        ax.scatter(tt, f1m, s=150, color=col, zorder=5, edgecolors="white", lw=1)
        ax.errorbar(tt, f1m, yerr=f1s, fmt="none", color=col, capsize=4, lw=1.5)
        ax.annotate(mn.replace("LogisticRegression","LR")
                       .replace("ComplementNB","CNB"),
                    (tt, f1m), textcoords="offset points",
                    xytext=(5,4), fontsize=8, color=col)

    ax.set_xlabel("Mean training time per fold (s)", fontsize=11)
    ax.set_ylabel("Macro-F1 (5-fold CV mean ± std)", fontsize=11)
    ax.set_title(f"Accuracy–Efficiency Tradeoff — {task}", fontweight="bold")
    ax.set_xscale("log")
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"efficiency.png"),
                  os.path.join(pdf_dir,"efficiency.pdf"))
    print("  ✓ Efficiency plot")

# ============================================================
# 6J. ERROR ANALYSIS
# ============================================================
def plot_error_analysis(fold0_dir, task, class_names, png_dir, pdf_dir):
    rows = []
    for mn in MODEL_ORDER:
        mid = safe_name(mn)
        ep  = os.path.join(fold0_dir,f"errors_{mid}.csv")
        if not os.path.exists(ep): continue
        err = pd.read_csv(ep)
        err["model"] = mn
        rows.append(err)
    if not rows: return
    all_err = pd.concat(rows, ignore_index=True)

    # URL length distribution of errors
    if "url" in all_err.columns:
        all_err["url_len"] = all_err["url"].astype(str).str.len()

    # TLD distribution of errors (top 10)
    fig, axes = plt.subplots(1, 2, figsize=(13,5))

    # Panel 1: error count per model
    ec = all_err.groupby("model").size().reindex(MODEL_ORDER).dropna()
    colors = [MODEL_COLORS.get(m,"#607D8B") for m in ec.index]
    axes[0].barh(ec.index, ec.values, color=colors)
    axes[0].set_xlabel("Error count (fold 0)")
    axes[0].set_title("Misclassification Count per Model", fontweight="bold")
    for i,(m,v) in enumerate(ec.items()):
        axes[0].text(v+20, i, f"{int(v):,}", va="center", fontsize=8)

    # Panel 2: TLD distribution in errors (HistGB)
    mid = safe_name("HistGB")
    hgb_err = all_err[all_err["model"]=="HistGB"]
    if "tld" in hgb_err.columns and len(hgb_err):
        tld_c = hgb_err["tld"].value_counts().head(12)
        axes[1].barh(tld_c.index[::-1], tld_c.values[::-1],
                     color=MODEL_COLORS["HistGB"])
        axes[1].set_xlabel("Error count")
        axes[1].set_title("HistGB Errors by TLD (top 12)", fontweight="bold")
        for i,(t,v) in enumerate(zip(tld_c.index[::-1], tld_c.values[::-1])):
            axes[1].text(v+1, i, str(int(v)), va="center", fontsize=8)
    else:
        axes[1].set_visible(False)

    fig.suptitle(f"Error Analysis — {task} (fold 0)", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"error_analysis.png"),
                  os.path.join(pdf_dir,"error_analysis.pdf"))

    # URL length distribution
    if "url_len" in all_err.columns:
        fig2, ax2 = plt.subplots(figsize=(8,5))
        for mn in MODEL_ORDER:
            sub = all_err[all_err["model"]==mn]["url_len"]
            if len(sub):
                ax2.hist(sub, bins=40, alpha=0.45, label=mn,
                         color=MODEL_COLORS.get(mn,"#607D8B"), density=True)
        ax2.set_xlabel("URL length (characters)")
        ax2.set_ylabel("Density")
        ax2.set_title("URL Length Distribution of Misclassified URLs", fontweight="bold")
        ax2.legend(fontsize=7)
        fig2.tight_layout()
        save_fig(fig2, os.path.join(png_dir,"error_url_length.png"),
                       os.path.join(pdf_dir,"error_url_length.pdf"))
    print("  ✓ Error analysis")

# ============================================================
# 6K. COMPREHENSIVE METRICS BAR CHART (CV means, all models)
# ============================================================
def plot_cv_metrics_comparison(cv_summary, task, png_dir, pdf_dir):
    SHOW = ["F1_Macro","MCC","Balanced_Accuracy","ROC_AUC_OVR","PR_AUC"]
    LABELS = ["Macro-F₁","MCC","Bal-Acc","ROC-AUC","PR-AUC"]
    mnames = [m for m in MODEL_ORDER if m in cv_summary]
    n_metrics = len(SHOW)
    n_models  = len(mnames)
    x = np.arange(n_metrics)
    w = 0.8/n_models

    fig, ax = plt.subplots(figsize=(12,5.5))
    for i,mn in enumerate(mnames):
        cv   = cv_summary[mn]
        vals = [cv.get(f"{m}_mean",np.nan) for m in SHOW]
        errs = [cv.get(f"{m}_std", 0)      for m in SHOW]
        bars = ax.bar(x + i*w - 0.4 + w/2, vals,
                      width=w, color=MODEL_COLORS.get(mn,"#607D8B"),
                      label=mn.replace("LogisticRegression","LR")
                               .replace("ComplementNB","CNB"),
                      yerr=errs, capsize=2,
                      error_kw={"elinewidth":0.8})

    ax.set_xticks(x); ax.set_xticklabels(LABELS, fontsize=10)
    ax.set_ylabel("Score (mean ± std, 5-fold CV)", fontsize=10)
    ax.set_title(f"TUMC {task} — All Models, All Metrics",
                 fontweight="bold", fontsize=12)
    ax.legend(ncol=4, fontsize=8, loc="lower right")
    lo = 0; hi = 1.05
    valid = [cv_summary[m].get(f"{SHOW[0]}_mean",np.nan) for m in mnames]
    if any(not np.isnan(v) for v in valid):
        lo = max(0, min(v for v in valid if not np.isnan(v)) - 0.08)
    ax.set_ylim(lo, hi)
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"cv_all_metrics.png"),
                  os.path.join(pdf_dir,"cv_all_metrics.pdf"))
    print("  ✓ CV metrics comparison")

# ============================================================
# 6L. FOLD-STABILITY PLOT (F1 per fold per model)
# ============================================================
def plot_fold_stability(model_fold_rows, task, png_dir, pdf_dir):
    mnames = [m for m in MODEL_ORDER if m in model_fold_rows]
    n_folds = len(next(iter(model_fold_rows.values())))
    fig, ax = plt.subplots(figsize=(9,5))
    for mn in mnames:
        rows = model_fold_rows[mn]
        f1s  = [r["F1_Macro"] for r in rows]
        flds = [r["Fold"]     for r in rows]
        col  = MODEL_COLORS.get(mn,"#607D8B")
        ax.plot(flds, f1s, "o-", color=col, label=mn, ms=5, lw=1.5)
        ax.fill_between(flds,
                        [np.mean(f1s)-np.std(f1s)]*n_folds,
                        [np.mean(f1s)+np.std(f1s)]*n_folds,
                        alpha=0.08, color=col)

    ax.set_xlabel("Fold"); ax.set_ylabel("Macro-F₁")
    ax.set_title(f"Cross-Fold Stability — {task}", fontweight="bold")
    ax.set_xticks(flds)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    save_fig(fig, os.path.join(png_dir,"fold_stability.png"),
                  os.path.join(pdf_dir,"fold_stability.pdf"))
    print("  ✓ Fold stability")

# ============================================================
# 7. COMBINED COMPARISON PLOT
# ============================================================
def make_comparison_plot(all_cv, out_dir):
    fig, axes = plt.subplots(1,2,figsize=(14,6))
    specs = [("5class","F1_Macro_mean","F1_Macro_std","Macro-F₁","Five-Class"),
             ("binary","ROC_AUC_OVR_mean","ROC_AUC_OVR_std","ROC-AUC","Binary")]
    for ax,(task,metric,std_col,ylabel,title) in zip(axes,specs):
        cv = all_cv.get(task,{})
        mns  = [m for m in MODEL_ORDER if m in cv]
        vals = [cv[m].get(metric,np.nan) for m in mns]
        errs = [cv[m].get(std_col,0)     for m in mns]
        cols = [MODEL_COLORS.get(m,"#90A4AE") for m in mns]
        x = np.arange(len(mns))
        bars = ax.bar(x, vals, color=cols, width=0.65,
                      yerr=errs, capsize=4,
                      error_kw={"elinewidth":1.5,"ecolor":"#333"})
        ax.set_xticks(x)
        ax.set_xticklabels([m.replace("LogisticRegression","LR")
                             .replace("ComplementNB","CNB") for m in mns],
                           rotation=30, ha="right")
        ax.set_ylabel(ylabel,fontsize=11)
        ax.set_title(f"{title} Detection — 5-Fold CV",fontweight="bold",fontsize=12)
        valid = [v for v in vals if not np.isnan(v)]
        if valid:
            ax.set_ylim(max(0,min(valid)-0.06), min(1.02,max(valid)+0.04))
        for bar,v in zip(bars,vals):
            if not np.isnan(v):
                ax.text(bar.get_x()+bar.get_width()/2,
                        bar.get_height()+0.002,f"{v:.3f}",
                        ha="center",va="bottom",fontsize=8)
        ax.spines[["top","right"]].set_visible(False)
        ax.grid(axis="y",alpha=0.3)
    fig.suptitle("TUMC: 7-Model Comparison (5-Fold Domain-Insulated CV)",
                 fontweight="bold",fontsize=13)
    plt.tight_layout(rect=[0,0.02,1,1])
    for ext in [".png",".pdf"]:
        fig.savefig(os.path.join(out_dir,f"comparison_7model_5fold{ext}"),
                    dpi=300 if ext==".png" else None,
                    bbox_inches="tight")
    plt.close(fig)
    print(f"  ✓ Combined comparison plot")

# ============================================================
# 8. LaTeX CV TABLE
# ============================================================
def make_cv_tex(cv_summary, task, out_path):
    mnames = [m for m in MODEL_ORDER if m in cv_summary]
    rows   = []
    for mn in mnames:
        d    = cv_summary[mn]
        f1m  = d.get("F1_Macro_mean",np.nan); f1s = d.get("F1_Macro_std",0)
        mccm = d.get("MCC_mean",np.nan);      mccs= d.get("MCC_std",0)
        aucm = d.get("ROC_AUC_OVR_mean",np.nan); aucs=d.get("ROC_AUC_OVR_std",0)
        prm  = d.get("PR_AUC_mean",np.nan)
        trt  = d.get("Train_Time_s_mean",np.nan)
        rows.append({
            "Model":          mn,
            "Macro-F$_1$":    f"${f1m:.3f}\\pm{f1s:.3f}$",
            "MCC":            f"${mccm:.3f}\\pm{mccs:.3f}$",
            "ROC-AUC":        f"${aucm:.4f}\\pm{aucs:.4f}$" if not np.isnan(aucm) else "---",
            "PR-AUC":         f"${prm:.3f}$"  if not np.isnan(prm)  else "---",
            "Train (s)":      f"{trt:.0f}"    if not np.isnan(trt)  else "---",
        })
    df = pd.DataFrame(rows)
    save_tex(df, out_path,
             caption=(f"Five-fold CV results for {task} task "
                      f"(135 features, domain-insulated folds). "
                      f"Mean $\\pm$ std reported across 5 folds. "
                      f"LogisticRegression uses StandardScaler; all tree models are scale-invariant."),
             label=f"tab:cv_{task}", index=False)
    print(f"  ✓ LaTeX CV table → {out_path}")

# ============================================================
# 9. LOAD DATA
# ============================================================
print("\n"+"="*70)
print(f"LOADING: {os.path.basename(INPUT_FILE)}")
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"  Rows: {len(df):,}  Folds: {sorted(df['fold'].unique())}")

FEATURES = [c for c in df.columns if c not in META_COLS]
if DROP_LEAKAGE:
    FEATURES = [c for c in FEATURES if c not in LEAKY_FEATURES]
FEATURES = list(dict.fromkeys(FEATURES))
for c in FEATURES:
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = pd.to_numeric(df[c],errors="coerce")
print(f"  Features: {len(FEATURES)}")

# ============================================================
# 10. MAIN LOOP
# ============================================================
all_tasks_cv = {}

for TASK in TASKS:
    target_col  = "label_enc" if TASK=="binary" else "class_enc"
    classes     = np.sort(df[target_col].dropna().unique())
    n_classes   = len(classes)
    is_bin      = n_classes==2
    class_names = get_class_names(df, TASK, classes)

    TASK_DIR = os.path.join(DATA_DIR, f"journal_eval_{TASK}")
    CV_DIR   = os.path.join(TASK_DIR, "cv_summary")
    CV_PNG   = os.path.join(CV_DIR,"figures_png")
    CV_PDF   = os.path.join(CV_DIR,"figures_pdf")
    makedirs(TASK_DIR, CV_DIR, CV_PNG, CV_PDF)

    print(f"\n{'='*70}")
    print(f"TASK: {TASK}  |  Classes: {n_classes}  |  Folds: {RUN_FOLDS}")

    model_fold_rows = {}   # model -> [row_f0, row_f1, ...]

    for TEST_FOLD in RUN_FOLDS:
        FOLD_DIR = os.path.join(TASK_DIR, f"fold{TEST_FOLD}")
        FCSV = os.path.join(FOLD_DIR,"csv"); FTEX = os.path.join(FOLD_DIR,"tex")
        FPNG = os.path.join(FOLD_DIR,"figures_png")
        FPDF = os.path.join(FOLD_DIR,"figures_pdf")
        makedirs(FOLD_DIR, FCSV, FTEX, FPNG, FPDF)

        train_df = df[df["fold"]!=TEST_FOLD]; test_df = df[df["fold"]==TEST_FOLD]
        X_tr=train_df[FEATURES].values; y_tr=train_df[target_col].values
        X_te=test_df[FEATURES].values;  y_te=test_df[target_col].values

        print(f"\n  ── Fold {TEST_FOLD}  (train={len(train_df):,}  test={len(test_df):,})")
        models = make_models(n_classes)

        fold_rows = []
        for mn, model in models.items():
            print(f"  {mn:24s}", end="")
            row = eval_one_fold(mn,model,X_tr,y_tr,X_te,y_te,
                                TASK,classes,class_names,test_df,
                                FCSV,FTEX,FPNG,FPDF,TEST_FOLD)
            fold_rows.append(row)
            model_fold_rows.setdefault(mn,[]).append(row)

        fold_df = pd.DataFrame(fold_rows).sort_values("F1_Macro",ascending=False)
        fold_df.to_csv(os.path.join(FCSV,"fold_summary.csv"),index=False)

        # Generate fold-0 detailed plots (expensive plots once only)
        if TEST_FOLD==0:
            print(f"\n  ── Generating detailed plots for fold 0 ──")
            plot_roc_curves(FCSV,TASK,classes,class_names,is_bin,FPNG,FPDF)
            plot_pr_curves(FCSV,TASK,classes,class_names,is_bin,FPNG,FPDF)
            plot_det_curves(FCSV,TASK,is_bin,FPNG,FPDF)
            plot_all_confusion_matrices(FCSV,TASK,classes,class_names,FPNG,FPDF)
            plot_calibration(FCSV,TASK,is_bin,FPNG,FPDF)
            plot_per_class_heatmap(FCSV,TASK,classes,class_names,is_bin,FPNG,FPDF)
            # threshold + cost (binary) or skip
            best_f1_model = fold_df.iloc[0]["Model"]
            plot_threshold_and_cost(FCSV,best_f1_model,is_bin,FPNG,FPDF)
            plot_error_analysis(FCSV,TASK,class_names,FPNG,FPDF)
            print(f"  Fold-0 detailed plots saved to {FPNG}")

    # ── CV aggregate ──────────────────────────────────────────
    print(f"\n  ── CV AGGREGATE ({TASK}) ──")
    task_cv = {}
    agg_rows = []
    for mn, rows in model_fold_rows.items():
        cv = aggregate_cv(rows)
        task_cv[mn] = cv
        agg_rows.append(cv)
        print(f"  {mn:24s}  F1={cv.get('F1_Macro_mean',0):.4f}"
              f"±{cv.get('F1_Macro_std',0):.4f}  MCC={cv.get('MCC_mean',0):.4f}")

    all_tasks_cv[TASK] = task_cv

    # CV summary CSV
    agg_df = pd.DataFrame(agg_rows).sort_values("F1_Macro_mean",ascending=False)
    agg_df.to_csv(os.path.join(CV_DIR,"cv_summary.csv"),index=False)

    # CV summary plots
    plot_cv_metrics_comparison(task_cv, TASK, CV_PNG, CV_PDF)
    plot_radar(task_cv, TASK, CV_PNG, CV_PDF)
    plot_efficiency(task_cv, TASK, CV_PNG, CV_PDF)
    plot_fold_stability(model_fold_rows, TASK, CV_PNG, CV_PDF)
    make_cv_tex(task_cv, TASK, os.path.join(CV_DIR,f"cv_summary_{TASK}.tex"))

# ── Combined comparison ───────────────────────────────────────
print(f"\n{'='*70}")
print("COMBINED COMPARISON PLOT")
make_comparison_plot(all_tasks_cv, COMBINED_DIR)

# ── Final summary ─────────────────────────────────────────────
print(f"\n{'='*70}  DONE  {'='*70}")
for task in TASKS:
    print(f"\n  [{task.upper()}]")
    cv  = all_tasks_cv.get(task,{})
    m_key = "F1_Macro_mean"
    for mn in sorted(cv, key=lambda m: cv[m].get(m_key,0), reverse=True):
        print(f"  {mn:24s}  F1={cv[mn].get(m_key,np.nan):.4f}"
              f"±{cv[mn].get('F1_Macro_std',0):.4f}"
              f"  MCC={cv[mn].get('MCC_mean',np.nan):.4f}"
              f"  AUC={cv[mn].get('ROC_AUC_OVR_mean',np.nan):.4f}")

print(f"""
Outputs:
  {DATA_DIR}/journal_eval_5class/
    fold0/ … fold4/  — per-fold confusion matrices, ROC, PR, errors
    cv_summary/      — radar, stability, efficiency, LaTeX table

  {DATA_DIR}/journal_eval_binary/
    (same structure)

  {DATA_DIR}/journal_eval_combined/
    comparison_7model_5fold.png/.pdf  — manuscript figure
""")

XGBoost  ✓
LightGBM ✓

LOADING: features_full_final_inductive_dedup.csv
  Rows: 1,239,308  Folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  Features: 135

TASK: 5class  |  Classes: 5  |  Folds: [0, 1, 2, 3, 4]

  ── Fold 0  (train=991,213  test=248,095)
  HistGB                    F1=0.9158  MCC=0.8792  AUC=0.9834  PR=0.9458  (498s)
  RandomForest              F1=0.8683  MCC=0.8244  AUC=0.9709  PR=0.9148  (167s)
  ExtraTrees                F1=0.8023  MCC=0.7254  AUC=0.9534  PR=0.8643  (108s)
  LogisticRegression        F1=0.8032  MCC=0.7492  AUC=0.9531  PR=0.8625  (468s)
  ComplementNB              F1=0.6632  MCC=0.5263  AUC=0.8570  PR=0.7059  (19s)
  XGBoost                   F1=0.8996  MCC=0.8562  AUC=0.9777  PR=0.9315  (173s)
  LightGBM                  F1=0.8927  MCC=0.8678  AUC=0.9821  PR=0.9418  (145s)

  ── Generating detailed plots for fold 0 ──
  ✓ ROC curves
  ✓ PR curves
  ✓ Combined confusion matrices
  ✓ Calibration curves
  ✓ Per-class heatmap
  